In [ ]:
# ── 1 · Импорты + модули ─────────────────────────────────────────
import os, importlib.util, logging
import numpy as np, pandas as pd
from scipy import stats
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('opt')
ROOT = os.path.dirname(os.getcwd())   # vol-engine/ (родитель 'jupyter strats'); портативно
def _load(name, path):
    s = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(s); s.loader.exec_module(m); return m
iopt = _load('intrinio_options', f'{ROOT}/intrinio_options.py')   # фетчер/фичи
il   = _load('intermarket_lab',  f'{ROOT}/intermarket_lab.py')    # bh_fdr/two_prop_p/wilson
MIN_N, Q = 40, 0.10
SYMBOL = ['SPX','SPXW']   # список → склейка SPX(мес)+SPXW(дневн/нед); SPY = 'SPY' строкой

In [2]:
# ── 2 · Дневные VRP/GEX фичи + таргеты (symbol-agnostic) ─────────
# Надёжный spot: SPY из кеша; SPX ≈ 10×SPY (прокси, точнее битого инференса);
# иначе — инференс из цепочки.
_spyf = f'{ROOT}/cache/intrinio/SPY_spot.parquet'
if SYMBOL == 'SPY' and os.path.exists(_spyf):
    spot_ser = pd.read_parquet(_spyf)['close']
elif SYMBOL == 'SPX' and os.path.exists(_spyf):
    spot_ser = pd.read_parquet(_spyf)['close'] * 10.0
else:
    spot_ser = None   # NDX и пр. — инференс (или подключим реальный spot)

of = iopt.daily_features(SYMBOL, spot_ser, recompute=(SYMBOL!='SPY'))  # SPX/NDX: вендорская IV/greeks битые → пересчёт из марок
log.info('%s: опционных дней %d  (%s .. %s)', SYMBOL, len(of),
         of.index.min().date() if len(of) else '-', of.index.max().date() if len(of) else '-')

ret = of['spot'].pct_change()
of['ret']         = ret
of['rv_21']       = iopt.realized_vol(of['spot'], 21)
of['vrp']         = of['atm_iv_30'] - of['rv_21']
of['iv_rv_ratio'] = of['atm_iv_30'] / of['rv_21']
of['gex_z']       = (of['gex'] - of['gex'].rolling(60).mean()) / of['gex'].rolling(60).std()
of['fwd_ret']     = ret.shift(-1)
of['fwd_absret']  = of['fwd_ret'].abs()
of['fwd_rv21']    = iopt.realized_vol(of['spot'], 21).shift(-21)
of['vrp_realized']= of['atm_iv_30'] - of['fwd_rv21']

if len(of) > 20:
    log.info('\n%s', of[['spot','atm_iv_30','rv_21','vrp','skew_25d','gex','gex_z']].describe().round(3).to_string())

# vol-term + credit (FRED) — макро-режим/кризис-контекст
try:
    _vc = il.vol_credit_features(str(of.index.min().date()), str(of.index.max().date()),
                                 'YOUR_FRED_API_KEY')
    of = of.join(_vc, how='left')
    for col in ['vix_term_slope','vix_term_ratio','hy_ig_spread','hy_oas_chg_5']:
        of[col] = of[col].ffill()
    log.info('vol/credit влиты: vix_term_slope покрытие %.0f%%', of['vix_term_slope'].notna().mean()*100)
except Exception as e:
    log.info('vc merge skip: %s', e)

./intrinio_options.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
17:45:52 INFO ['SPX', 'SPXW']: опционных дней 1126  (2021-09-27 .. 2026-06-15)
17:45:52 INFO 
           spot  atm_iv_30     rv_21       vrp  skew_25d           gex     gex_z
count  1126.000   1126.000  1105.000  1105.000  1125.000  1.125000e+03  1007.000
mean   5230.009      0.160     0.154     0.006     0.227  8.575386e+09    -0.019
std    1045.504      0.052     0.075     0.054     0.693  3.635215e+10     1.186
min    3585.000      0.000     0.056    -0.328    -0.081 -8.759021e+10    -3.236
25%    4350.000      0.122     0.105    -0.012     0.036 -1.726863e+10    -0.922
50%    5067.500      0.145   

In [3]:
# ── 3 · VRP: существует ли премия и предсказуема ли ──────────────
# Классический edge: IV систематически > будущей realized vol → шорт vol платит.
v = of.dropna(subset=['vrp', 'vrp_realized'])
if len(v) >= MIN_N:
    log.info('VRP премия (IV − forward RV): mean=%.4f  доля>0=%.1f%%  n=%d',
             v['vrp_realized'].mean(), (v['vrp_realized'] > 0).mean()*100, len(v))
    log.info('Spearman(vrp_t, vrp_realized) = %.3f  (предсказывает ли текущий VRP будущий gap)',
             spearmanr(v['vrp'], v['vrp_realized']).correlation)
    # bucket VRP -> forward realized gap
    v = v.copy(); v['vrp_b'] = pd.qcut(v['vrp'], 5, labels=False, duplicates='drop')
    log.info('VRP-квинтиль → forward (IV−RV):\n%s',
             v.groupby('vrp_b')['vrp_realized'].agg(['mean','count']).round(4).to_string())
else:
    log.info('⏳ VRP: мало данных (%d), ждём закачку', len(v))

17:45:55 INFO VRP премия (IV − forward RV): mean=0.0046  доля>0=64.1%  n=1084
17:45:55 INFO Spearman(vrp_t, vrp_realized) = -0.047  (предсказывает ли текущий VRP будущий gap)
17:45:55 INFO VRP-квинтиль → forward (IV−RV):
         mean  count
vrp_b               
0      0.0062    217
1      0.0071    217
2      0.0029    216
3     -0.0008    217
4      0.0078    217


In [4]:
# ── 4 · GEX → next-day волатильность (FDR по квинтилям) ──────────
g = of.dropna(subset=['gex_z', 'fwd_absret'])
if len(g) >= 5*MIN_N:
    g = g.copy(); g['gex_b'] = pd.qcut(g['gex_z'], 5, labels=False, duplicates='drop')
    rows = []
    for k, grp in g.groupby('gex_b'):
        n = len(grp); rest = g['fwd_absret'].drop(grp.index)
        _, pmw = stats.mannwhitneyu(grp['fwd_absret'], rest, alternative='two-sided')
        rows.append({'gex_quintile': int(k), 'n': n,
                     'next_absret_%': round(grp['fwd_absret'].mean()*100, 3),
                     'next_autocorr': round(grp['ret'].corr(grp['fwd_ret']), 3), 'p_mw': pmw})
    gd = pd.DataFrame(rows); gd['q_mw'] = il.bh_fdr(gd['p_mw'].values)
    log.info('GEX-квинтиль → next-day |ret| (отриц. GEX = ожидаем экспансию):\n%s',
             gd.to_string(index=False))
else:
    log.info('⏳ GEX: мало данных (%d), ждём закачку', len(g))

17:45:55 INFO GEX-квинтиль → next-day |ret| (отриц. GEX = ожидаем экспансию):
 gex_quintile   n  next_absret_%  next_autocorr         p_mw         q_mw
            0 202          0.932          0.055 2.530747e-05 4.217912e-05
            1 201          1.044         -0.134 3.848806e-07 9.622015e-07
            2 201          0.734          0.065 9.285271e-01 9.285271e-01
            3 201          0.623         -0.137 7.543089e-04 9.428861e-04
            4 201          0.486          0.017 1.777185e-09 8.885925e-09


In [5]:
# ── 5 · Инкрементальность (нестед): naive → +VRP → +GEX ──────────
VRP_BLK = ['vrp', 'iv_rv_ratio', 'atm_iv_30', 'iv_term_slope', 'skew_25d']
GEX_BLK = ['gex_z']
def _nested(blocks, y):
    idx = y.dropna().index
    for _, X in blocks: idx = idx.intersection(X.dropna().index)
    if len(idx) < 300:
        return None
    months = pd.period_range(idx.min(), idx.max(), freq='M'); out = {}
    for name, X in blocks:
        Xv, yv = X.loc[idx], y.loc[idx]; pred = pd.Series(np.nan, index=idx)
        for per in months:
            m0 = per.to_timestamp(); m1 = (per+1).to_timestamp()
            tr = Xv.index[Xv.index < m0]; te = Xv.index[(Xv.index >= m0) & (Xv.index < m1)]
            if len(tr) < 252 or len(te) == 0: continue
            sc = StandardScaler().fit(Xv.loc[tr])
            rg = Ridge(alpha=1.0).fit(sc.transform(Xv.loc[tr]), yv.loc[tr])
            pred.loc[te] = rg.predict(sc.transform(Xv.loc[te]))
        m = pred.dropna().index; yt, pp = yv.loc[m], pred.loc[m]
        sse = ((yt-pp)**2).sum(); sst = ((yt-yt.mean())**2).sum()
        out[name] = {'R2': 1-sse/sst, 'Spearman': spearmanr(pp, yt).correlation, 'n': len(m)}
    return out

y = of['fwd_absret']
naive = of['rv_21'].rename('rv').to_frame()
blocks = [('M0_naive', naive),
          ('M1_+VRP',  pd.concat([naive, of[VRP_BLK]], axis=1)),
          ('M2_+GEX',  pd.concat([naive, of[VRP_BLK], of[GEX_BLK]], axis=1))]
r = _nested(blocks, y)
if r:
    rdf = pd.DataFrame(r).T
    log.info('VOL next-day · нестед инкрементальность:\n%s', rdf.round(4).to_string())
    log.info('ΔR² VRP (M1-M0): %+.4f   ΔR² GEX (M2-M1): %+.4f',
             r['M1_+VRP']['R2'] - r['M0_naive']['R2'],
             r['M2_+GEX']['R2'] - r['M1_+VRP']['R2'])
else:
    log.info('⏳ инкрементальность: нужно >300 дней с полными фичами, ждём закачку')

17:45:55 INFO VOL next-day · нестед инкрементальность:
              R2  Spearman      n
M0_naive  0.0240    0.1334  735.0
M1_+VRP   0.1563    0.2742  735.0
M2_+GEX   0.1618    0.3117  735.0
17:45:55 INFO ΔR² VRP (M1-M0): +0.1323   ΔR² GEX (M2-M1): +0.0055


In [6]:
# ── 6 · Направленный edge VRP/skew/GEX (FDR, ожидаемо слабо) ────
dd = of.dropna(subset=['fwd_ret', 'vrp', 'skew_25d', 'gex_z'])
if len(dd) >= 5*MIN_N:
    base_wr = (dd['fwd_ret'] > 0).mean(); tot_w = int((dd['fwd_ret']>0).sum()); tot_n = len(dd)
    rows = []
    for feat in ['vrp', 'skew_25d', 'gex_z']:
        dd = dd.copy(); dd['_b'] = pd.qcut(dd[feat], 5, labels=False, duplicates='drop')
        for k, grp in dd.groupby('_b'):
            n = len(grp)
            if n < MIN_N: continue
            kk = int((grp['fwd_ret'] > 0).sum())
            rows.append({'feat': feat, 'q': int(k), 'n': n,
                         'wr': round((grp['fwd_ret']>0).mean()*100, 1),
                         'next_ret_%': round(grp['fwd_ret'].mean()*100, 3),
                         'p': il.two_prop_p(kk, n, tot_w-kk, tot_n-n)})
    ed = pd.DataFrame(rows); ed['q_fdr'] = il.bh_fdr(ed['p'].values)
    log.info('Направление next-day (baseline WR=%.1f%%), значимых q<%.2f: %d\n%s',
             base_wr*100, Q, int((ed['q_fdr']<Q).sum()), ed.sort_values('p').head(20).to_string(index=False))
else:
    log.info('⏳ направление: мало данных (%d), ждём закачку', len(dd))

17:45:55 INFO Направление next-day (baseline WR=51.9%), значимых q<0.10: 0
    feat  q   n   wr  next_ret_%        p    q_fdr
skew_25d  1 201 57.2       0.107 0.091182 0.683864
   gex_z  2 201 57.2       0.163 0.091182 0.683864
skew_25d  3 201 47.3      -0.017 0.142358 0.711791
   gex_z  3 201 47.8      -0.021 0.190449 0.714183
     vrp  3 201 55.2       0.005 0.290081 0.870243
     vrp  4 201 49.3       0.062 0.403260 0.976766
   gex_z  1 201 49.8      -0.068 0.497772 0.976766
     vrp  2 201 50.2       0.113 0.602933 0.976766
   gex_z  4 201 53.2       0.100 0.669604 0.976766
     vrp  0 202 53.0       0.055 0.730723 0.976766
skew_25d  4 201 51.2       0.035 0.837915 0.976766
   gex_z  0 202 51.5       0.071 0.897836 0.976766
     vrp  1 201 51.7       0.009 0.962714 0.976766
skew_25d  2 201 51.7       0.044 0.962714 0.976766
skew_25d  0 202 52.0       0.076 0.976766 0.976766


In [7]:
# ── 7 · Matched-horizon: GEX→next-day ХВОСТ; 90DTE/OI→21d vol ────
# (a) Near-term GEX(0-7d) → P(|ret| > 80-перцентиль) next-day (хвостовой таргет)
thr = of['fwd_absret'].quantile(0.80)
gg = of.dropna(subset=['gex_z', 'fwd_absret']).copy()
if len(gg) >= 5*MIN_N:
    gg['big'] = (gg['fwd_absret'] > thr).astype(int)
    gg['b']   = pd.qcut(gg['gex_z'], 5, labels=False, duplicates='drop')
    tot_big = int(gg['big'].sum())
    rows = []
    for k, grp in gg.groupby('b'):
        n = len(grp); kk = int(grp['big'].sum())
        rows.append({'gex_z_q': int(k), 'n': n, 'P_big_%': round(grp['big'].mean()*100, 1),
                     'p': il.two_prop_p(kk, n, tot_big-kk, len(gg)-n)})
    t = pd.DataFrame(rows); t['q_fdr'] = il.bh_fdr(t['p'].values)
    log.info('Near-term GEX(0-7d) → P(|ret|>%.2f%%) next-day (хвост):\n%s', thr*100, t.to_string(index=False))
else:
    log.info('⏳ GEX-хвост: мало данных (%d)', len(gg))

# (b) 90DTE/OI позиционирование → forward-vol на НЕперекрывающихся 21д блоках
#     (предиктор = позиционка в начале блока; таргет = realized vol внутри блока)
from scipy.stats import spearmanr
H = 21
ofx = of.dropna(subset=['ret']).copy()
ofx['blk'] = np.arange(len(ofx)) // H
gb = ofx.groupby('blk')
blk_vol = gb['ret'].std() * np.sqrt(252)          # realized vol блока (forward от старта)
rows = []
for col in ['put_call_oi','call_wall_dist','put_wall_dist','oi_concentration','gex_z','vrp']:
    pred = gb[col].first()
    m = pred.notna() & blk_vol.notna()
    if m.sum() < 15: continue
    r, pv = spearmanr(pred[m], blk_vol[m])
    rows.append({'feature': col, 'n_blocks': int(m.sum()),
                 'spearman_vs_fwdvol': round(r, 3), 'p': round(pv, 3)})
log.info('Позиционирование → forward-vol (НЕперекрыв. %dд блоки):\n%s', H,
         pd.DataFrame(rows).to_string(index=False))

17:45:55 INFO Near-term GEX(0-7d) → P(|ret|>1.19%) next-day (хвост):
 gex_z_q   n  P_big_%        p    q_fdr
       0 202     29.2 0.000168 0.000281
       1 201     29.9 0.000062 0.000154
       2 201     17.9 0.456666 0.456666
       3 201     12.4 0.003481 0.004351
       4 201      9.5 0.000040 0.000154
17:45:55 INFO Позиционирование → forward-vol (НЕперекрыв. 21д блоки):
         feature  n_blocks  spearman_vs_fwdvol     p
     put_call_oi        54              -0.145 0.297
  call_wall_dist        54               0.290 0.033
   put_wall_dist        54               0.201 0.145
oi_concentration        54              -0.381 0.004
           gex_z        50              -0.379 0.007
             vrp        54               0.085 0.539


In [8]:
# ── 8 · Субпериодная стабильность GEX(0-7d)→хвост (по годам) ──────
thr = of['fwd_absret'].quantile(0.80)
gg = of.dropna(subset=['gex_z','fwd_absret']).copy()
gg['big'] = (gg['fwd_absret'] > thr).astype(int)
gg['yr']  = gg.index.year
rows = []
for yr, g in gg.groupby('yr'):
    if len(g) < 60: continue
    g = g.copy(); g['b'] = pd.qcut(g['gex_z'], 3, labels=False, duplicates='drop')
    pb = g.groupby('b')['big'].mean() * 100
    sp = g[['gex_z','big']].corr(method='spearman').iloc[0,1]
    rows.append({'year': yr, 'n': len(g),
                 'lowGEX_Pbig%': round(pb.get(0, np.nan),1),
                 'highGEX_Pbig%': round(pb.get(pb.index.max(), np.nan),1),
                 'spread': round(pb.get(0,np.nan) - pb.get(pb.index.max(),np.nan),1),
                 'spearman': round(sp,3)})
log.info('GEX(0-7d)→хвост по годам (стабильно если spread>0 и spearman<0 каждый год):\n%s',
         pd.DataFrame(rows).to_string(index=False))

17:45:55 INFO GEX(0-7d)→хвост по годам (стабильно если spread>0 и spearman<0 каждый год):
 year   n  lowGEX_Pbig%  highGEX_Pbig%  spread  spearman
 2022 212          46.5           32.4    14.1    -0.132
 2023 178          16.9            8.3     8.6    -0.081
 2024 247          23.2            2.4    20.8    -0.274
 2025 250          26.5            9.5    17.0    -0.240
 2026 112          31.6           10.5    21.1    -0.183


In [9]:
# ── 9 · VRP-harvest + GEX-filter · P&L (РЕАЛЬНЫЙ straddle спред) ──
d = of.dropna(subset=['atm_iv_30','fwd_ret']).copy()
d['iv_day2'] = (d['atm_iv_30']/np.sqrt(252))**2
d['premium'] = d['iv_day2'] - d['fwd_ret']**2
d['spread']  = d['straddle_spread'].fillna(d['straddle_spread'].median())
log.info('Реальный round-trip спред ATM-straddle: median=%.2f%% (vs выдуманные 10%%)', d['spread'].median()*100)
gex_hi  = (d['gex_z'] >= d['gex_z'].median()).fillna(False)
vrp_pos = (d['vrp'] > 0).fillna(False)
sigs = {'always': pd.Series(1.0,index=d.index), 'VRP': vrp_pos.astype(float), 'VRP+GEX': (vrp_pos&gex_hi).astype(float)}
def _stats(pos):
    cost = d['spread']*d['iv_day2']*pos.diff().abs().fillna(pos)
    pnl  = pos*d['premium'] - cost
    sh   = pnl.mean()/pnl.std()*np.sqrt(252) if pnl.std()>0 else 0
    return {'mean_bp':round(pnl.mean()*1e4,2),'Sharpe':round(sh,2),'active_%':round((pos!=0).mean()*100,0),
            'worst_bp':round(pnl.min()*1e4,1),'total_bp':round(pnl.sum()*1e4,0)}
log.info('VRP-harvest P&L (реальный спред):\n%s', pd.DataFrame({k:_stats(v) for k,v in sigs.items()}).T.to_string())

17:45:55 INFO Реальный round-trip спред ATM-straddle: median=1.38% (vs выдуманные 10%)
17:45:55 INFO VRP-harvest P&L (реальный спред):
         mean_bp  Sharpe  active_%  worst_bp  total_bp
always     -0.05   -0.22     100.0     -94.1     -55.0
VRP        -0.07   -0.34      63.0     -94.1     -81.0
VRP+GEX     0.02    0.45      24.0     -18.4      26.0


In [10]:
# ── 10 · Варианты × стресс по косту (×1/2/3 реального straddle-спреда) ──
d = of.dropna(subset=['atm_iv_30','fwd_ret']).copy()
d['iv_day2'] = (d['atm_iv_30']/np.sqrt(252))**2
d['premium'] = d['iv_day2'] - d['fwd_ret']**2
d['spread']  = d['straddle_spread'].fillna(d['straddle_spread'].median())
gex_hi  = (d['gex_z'] >= d['gex_z'].median()).fillna(False)
vrp_pos = (d['vrp'] > 0).fillna(False)
vrp_sz  = d['vrp'].rank(pct=True).fillna(0.0)
macro_safe = (d['vix_term_slope'] <= 0).fillna(True) if 'vix_term_slope' in d.columns else pd.Series(True,index=d.index)
variants = {'always':pd.Series(1.0,index=d.index), 'GEX_only':gex_hi.astype(float), 'VRP_only':vrp_pos.astype(float),
            'VRP+GEX':(vrp_pos&gex_hi).astype(float), 'VRPsize x GEX':vrp_sz*gex_hi.astype(float),
            'GEX+macro':(gex_hi&macro_safe).astype(float), 'VRPsz x GEX x macro':vrp_sz*(gex_hi&macro_safe).astype(float)}
def _m(pos, mult):
    cost = mult*d['spread']*d['iv_day2']*pos.diff().abs().fillna(pos); pnl = pos*d['premium']-cost
    sh = pnl.mean()/pnl.std()*np.sqrt(252) if pnl.std()>0 else 0
    return sh, pnl.min()*1e4
tbl=[]
for name,pos in variants.items():
    r={'variant':name,'active%':round((pos>0).mean()*100,0)}
    for mlt in [1,2,3]: r[f'Sh@{mlt}xspr']=round(_m(pos,mlt)[0],2)
    r['worst_bp']=round(_m(pos,1)[1],1); tbl.append(r)
log.info('Стратегии × стресс по косту (median спред=%.2f%%, ×1/2/3):\n%s',
         d['spread'].median()*100, pd.DataFrame(tbl).to_string(index=False))

17:45:55 INFO Стратегии × стресс по косту (median спред=1.38%, ×1/2/3):
            variant  active%  Sh@1xspr  Sh@2xspr  Sh@3xspr  worst_bp
             always    100.0     -0.22     -0.22     -0.22     -94.1
           GEX_only     45.0      0.97      0.93      0.89     -18.4
           VRP_only     63.0     -0.34     -0.35     -0.37     -94.1
            VRP+GEX     24.0      0.45      0.42      0.39     -18.4
      VRPsize x GEX     45.0      0.81      0.77      0.73      -8.6
          GEX+macro     44.0      1.02      0.98      0.95     -18.4
VRPsz x GEX x macro     44.0      0.92      0.88      0.84      -8.6


In [11]:
# ── 11 · Дилер vs ретейл: positioning → forward-vol (неперекрыв. блоки) ──
from scipy.stats import spearmanr
INSTR = {'SPY':'SPY', 'QQQ':'QQQ', 'SPX':['SPX','SPXW'], 'NDX':['NDX','NDXP']}
POSF  = ['oi_concentration','gex_z','call_wall_dist','put_wall_dist','put_call_oi']
H = 21
comp = {}
for name, sym in INSTR.items():
    o = iopt.daily_features(sym, recompute=(name in ('SPX','NDX')))
    if o.empty: continue
    o['ret']   = o['spot'].pct_change()
    o['gex_z'] = (o['gex'] - o['gex'].rolling(60).mean()) / o['gex'].rolling(60).std()
    o = o.dropna(subset=['ret'])
    o['blk'] = np.arange(len(o)) // H
    gb = o.groupby('blk'); vol = gb['ret'].std()*np.sqrt(252)
    row = {'n_blk': vol.notna().sum()}
    for col in POSF:
        if col not in o.columns: row[col] = np.nan; continue
        pr = gb[col].first(); m = pr.notna() & vol.notna()
        row[col] = round(spearmanr(pr[m], vol[m]).correlation, 3) if m.sum() >= 15 else np.nan
    comp[name] = row
    log.info('  %s готов', name)
log.info('Spearman positioning→forward-vol (неперекрыв. %dд): дилер SPX/NDX vs ретейл SPY/QQQ\n%s',
         H, pd.DataFrame(comp).T.to_string())

./intrinio_options.py:112: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


KeyboardInterrupt: 

In [12]:
# ── 12 · Стресс-эпизоды: уводит ли GEX/macro-гейт из позиции в худшие дни ──
d = of.dropna(subset=['atm_iv_30','fwd_ret']).copy()
d['iv_day2'] = (d['atm_iv_30']/np.sqrt(252))**2
d['premium'] = d['iv_day2'] - d['fwd_ret']**2
gex_hi = (d['gex_z'] >= d['gex_z'].median()).fillna(False)
macro_safe = (d['vix_term_slope'] <= 0).fillna(True) if 'vix_term_slope' in d.columns else pd.Series(True, index=d.index)
d['pos'] = (gex_hi & macro_safe).astype(float)   # 1 = шортим vol; 0 = гейт увёл (flat)
d['absret'] = d['fwd_ret'].abs()

worst = d.nlargest(15, 'absret')[['absret','fwd_ret','gex_z','vix_term_slope','pos','premium']]
log.info('15 худших дней (short-vol боль); pos=0 = гейт увёл в FLAT:\n%s', worst.round(4).to_string())
log.info('Доля худших дней с FLAT-гейтом: %.0f%%   P&L в них: naive=%.0fbp  gated=%.0fbp',
         (worst['pos']==0).mean()*100, worst['premium'].sum()*1e4, (worst['pos']*worst['premium']).sum()*1e4)

win = d.loc['2024-07-29':'2024-08-09', ['fwd_ret','gex_z','vix_term_slope','pos','premium']]
if len(win):
    log.info('Окно 2024-07-29..08-09 (yen carry unwind, VIX~65):\n%s', win.round(4).to_string())

17:46:17 INFO 15 худших дней (short-vol боль); pos=0 = гейт увёл в FLAT:
            absret  fwd_ret   gex_z  vix_term_slope  pos  premium
date                                                             
2025-04-08  0.1007   0.1007 -0.8970           10.83  0.0  -0.0094
2025-04-03  0.0613  -0.0613 -1.2147            2.64  0.0  -0.0035
2025-04-02  0.0461  -0.0461 -0.3329            0.06  0.0  -0.0020
2022-09-12  0.0449  -0.0449  0.4336           -1.50  1.0  -0.0018
2022-05-17  0.0440  -0.0440 -0.5463           -2.27  0.0  -0.0017
2022-06-10  0.0384  -0.0384 -1.3796           -1.97  0.0  -0.0014
2025-04-09  0.0366  -0.0366 -0.3542            3.70  0.0  -0.0011
2022-04-25  0.0360  -0.0360 -1.1120           -2.05  0.0  -0.0011
2022-08-25  0.0358  -0.0358  0.6453           -2.35  1.0  -0.0011
2022-10-28  0.0357   0.0357  0.2189           -1.15  1.0  -0.0010
2022-06-23  0.0344   0.0344 -0.1876           -1.14  0.0  -0.0009
2025-05-09  0.0336   0.0336  0.9060           -1.48  1.0  -0.0010
202

In [13]:
# ── 13 · Risk/sizing слой: FORWARD vol-target (inverse-IV) + forward-гейты ──
# ВЫВОД (доказано перебором): для short-vol правильный sizing — FORWARD по IV-уровню,
#   не trailing realized-vol. Все backward-looking контроли (vol-target по реализ. воле,
#   непрерывный DD-cap, equity-based kill, DD-floor) реагируют ПОСЛЕ удара и душат
#   пост-стресс восстановление (богатейший период для short-vol) → нетто-минус.
#   Просадки тут = одиночные GAP-дни; backward-контроль предотвратить их не может. Crisis-off
#   делают FORWARD-гейты (GEX+macro). А VRP-вес при IV-сайзинге избыточен и вреден → выкинут.
# ДИЗАЙН: pos = gate(GEX & macro_safe) · sz_iv,  sz_iv = (REF_IV / IV)².
#   REF_IV = риск-диал: НЕ влияет на Sharpe (~2.1), задаёт ср. плечо и maxDD линейно.
#   Point-in-time: IV_t известна на входе (хол t → t+1). L — масштаб капитала (нормировка
#   неуправляемой базы к TARGET_VOL; при деплое = решение по размеру капитала, не фит).
TARGET_VOL = 0.15            # годовой риск неуправляемой базы (задаёт масштаб L)
K_MAX      = 3.0             # потолок плеча на сайзере
COST_MULT  = 1.0             # реальный straddle-спред ×1
REF_GRID   = [0.12, 0.13, 0.14, 0.16, 0.18, 0.20]   # риск-диал REF_IV (весь диапазон)

d = of.dropna(subset=['atm_iv_30','fwd_ret']).copy()
d['iv_day2'] = (d['atm_iv_30']/np.sqrt(252))**2
d['premium'] = d['iv_day2'] - d['fwd_ret']**2
d['spread']  = d['straddle_spread'].fillna(d['straddle_spread'].median())
gex_hi  = (d['gex_z'] >= d['gex_z'].median()).fillna(False)
vrp_sz  = d['vrp'].rank(pct=True).fillna(0.0)
macro_safe = (d['vix_term_slope'] <= 0).fillna(True) if 'vix_term_slope' in d.columns else pd.Series(True, index=d.index)
gate = (gex_hi & macro_safe).astype(float)
# БОЕВОЙ = flat GEX+macro гейт. VRP-вес ОТКЛЮЧЁН: при forward IV-сайзинге он избыточен и вредит
#   (Sharpe 1.88 vs flat 2.10, worst −13.4% vs −8.8%). VRP остаётся причиной шортить волу
#   (премия >0 в 64% дней), но НЕ как size-вес. Калибровка ниже: vrp_sz→premium немонотонна.
pos_raw = gate.clip(0, 1)                     # боевой сигнал ∈ {0,1}
pos_vrp = (vrp_sz * gate).clip(0, 1)          # отвергнутый VRP-взвешенный (для сравнения)
iv = d['atm_iv_30'].clip(lower=0.05)

# масштаб капитала L: неуправляемая база (pos_raw, без сайзера) → вола = TARGET_VOL
base_raw = pos_raw*d['premium'] - COST_MULT*d['spread']*d['iv_day2']*pos_raw.diff().abs().fillna(pos_raw)
L = TARGET_VOL/(base_raw.std()*np.sqrt(252))
prem = (d['premium']*L).values; ivd = (d['iv_day2']*L).values; spr = d['spread'].values
n = len(d)

def backtest(pos_series, ref_iv):
    sz = ((ref_iv/iv)**2).clip(0, K_MAX).values    # FORWARD: IV высокая → размер вниз ДО шторма
    praw = pos_series.values; prev = 0.0; pnl = np.empty(n); lev = np.empty(n)
    for t in range(n):
        pos  = float(np.clip(praw[t]*sz[t], 0, K_MAX))
        pnl[t] = pos*prem[t] - COST_MULT*spr[t]*ivd[t]*abs(pos - prev)
        lev[t] = pos; prev = pos
    return pd.Series(pnl, index=d.index), pd.Series(lev, index=d.index)

def metrics(s):
    eq = 1 + s.cumsum(); dd = eq/eq.cummax() - 1
    ar = s.mean()*252; av = s.std()*np.sqrt(252); mdd = dd.min()
    return dict(annRet=f'{ar*100:.1f}%', annVol=f'{av*100:.1f}%', Sharpe=round(ar/av, 2) if av > 0 else 0,
                maxDD=f'{mdd*100:.1f}%', Calmar=round(ar/abs(mdd), 2) if mdd < 0 else np.nan,
                worst=f'{s.min()*100:.1f}%')

# (a) Боевой gate × весь риск-диал REF_IV
tbl = {'raw (no sizing)': {**metrics(base_raw*L), 'avgLev': 1.00}}
for rv in REF_GRID:
    s, lev = backtest(pos_raw, rv); m = metrics(s); m['avgLev'] = round(lev[lev > 0].mean(), 2)
    tbl[f'gate · REF={rv:g}'] = m
log.info('FORWARD IV-sizing на flat GEX+macro (L=%.0f×; REF_IV=риск-диал, Sharpe~2.1 инвариантен):\n%s',
         L, pd.DataFrame(tbl).T.to_string())

# (b) почему VRP-вес выкинут: flat gate vs VRPsz×gate при опорном REF_IV
REF0 = 0.14
cmp = {}
for nm, ps in [('flat gate (БОЕВОЙ)', pos_raw), ('VRPsz×gate (отвергнут)', pos_vrp)]:
    s, _ = backtest(ps, REF0); cmp[nm] = metrics(s)
log.info('VRP-вес vs flat (REF_IV=%.2f): flat лучше → VRP убран из sizing:\n%s', REF0,
         pd.DataFrame(cmp).T.to_string())

# (c) Калибровка: vrp_sz→premium на gated-днях (немонотонна = VRP-вес не оправдан)
g  = pos_raw > 0
cal = pd.DataFrame({'vrp_q': pd.qcut(vrp_sz[g], 5, labels=False, duplicates='drop'),
                    'prem_bp': d['premium'][g]*1e4})
log.info('Калибровка vrp_sz → premium (gated, bp):\n%s',
         cal.groupby('vrp_q')['prem_bp'].agg(['mean', 'count']).round(2).to_string())

18:41:22 INFO FORWARD IV-sizing на flat GEX+macro (L=91×; REF_IV=риск-диал, Sharpe~2.1 инвариантен):
                annRet annVol Sharpe   maxDD Calmar   worst avgLev
raw (no sizing)  15.3%  15.0%   1.02  -34.4%   0.44  -16.8%    1.0
gate · REF=0.12  15.5%   7.5%   2.06  -12.9%    1.2   -6.5%   0.87
gate · REF=0.13  18.3%   8.8%   2.08  -15.0%   1.22   -7.6%   1.02
gate · REF=0.14  21.4%  10.2%    2.1  -17.2%   1.24   -8.8%   1.18
gate · REF=0.16  28.2%  13.2%   2.13  -21.9%   1.29  -11.6%   1.53
gate · REF=0.18  35.8%  16.7%   2.15  -27.0%   1.33  -14.6%   1.92
gate · REF=0.2   42.4%  20.2%    2.1  -32.3%   1.31  -18.0%   2.25
18:41:22 INFO VRP-вес vs flat (REF_IV=0.14): flat лучше → VRP убран из sizing:
                       annRet annVol Sharpe   maxDD Calmar  worst
flat gate (БОЕВОЙ)      21.4%  10.2%    2.1  -17.2%   1.24  -8.8%
VRPsz×gate (отвергнут)   9.6%   5.1%   1.88   -8.4%   1.14  -6.0%
18:41:22 INFO Калибровка vrp_sz → premium (gated, bp):
       mean  count
vrp_q       